# Lecture 4.3 — Streaming Raw Text Deltas vs Structured Run Item Events

**Section 04 — Running Agents, Results & Streaming**

In this notebook you'll go deeper on the raw event layer that powers `Runner.run_streamed()`. You'll see the full Responses API event lifecycle, stream text token by token, stream function call argument deltas in real time, stream reasoning tokens for GPT-5 models, and build a clear decision guide for choosing raw events vs run item events.

## Cell 1 — Install the OpenAI Agents SDK

📌 **Notebook update notice:** this lecture's markdown references `openai-agents==0.18.0` as the pinned version. Since recording, a downstream dependency change (`openai>=2.45.0`, released July 9, 2026) broke `openai-agents` versions below 0.18.1 — `Runner.run()` will fail on the version stated above. This notebook has been updated to pin `openai-agents==0.18.3`, which fixes the issue without changing any of the code or concepts taught in the lecture. Please use the version pinned below, not the one mentioned in the recording.

This notebook uses the OpenAI Agents SDK for Python. The cell below installs it.

The version is pinned so that the examples in this notebook behave exactly as shown, regardless of when you're watching this. If you already have the package installed in this session, running the cell again is harmless, it just confirms the version and moves on.

In [1]:
# Pinned for reproducibility. Updated after recording — see the
# notice above. Originally pinned to 0.18.0 as stated in the
# video; updated to 0.18.3 to fix a breaking change introduced
# by openai>=2.45.0 (July 9, 2026).
# To use the latest version instead, run: pip install openai-agents
!pip install openai-agents==0.18.3 -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 880.8/880.8 kB 18.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 142.6/142.6 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.4/223.4 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.4/69.4 kB 5.0 MB/s eta 0:00:00


## Cell 2 — Configure Your OpenAI API Key

This notebook uses Google Colab Secrets to store your API key securely, rather than typing it directly into a cell.

**Steps to add your secret in Colab:**
1. Click the key icon in the left sidebar of Colab.
2. Click **Add new secret**.
3. Set the name to `OPENAI_API_KEY`.
4. Paste your API key as the value.
5. Toggle **Notebook access** on for this notebook.

**Local users:** if you're running this outside Colab, set the `OPENAI_API_KEY` environment variable in your terminal before starting your script, instead of using `userdata.get()`.

In [2]:
from google.colab import userdata
import os

os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")

## Cell 3 — Set the Model Name

We declare `MODEL_NAME` once here and use it everywhere an `Agent` needs a model string in this notebook. Changing this one variable updates the model used across the entire notebook.

In [3]:
# See latest models at: https://platform.openai.com/docs/models
MODEL_NAME = "gpt-5.4-mini"

## Cell 4 — Imports

This lecture introduces several new imports beyond what you've used so far, so it's worth pausing on where each one comes from.

All of the raw streaming event classes below come from `openai.types.responses`, because raw events are the underlying OpenAI Responses API protocol, while the `agents` package only wraps them.

| Import | Source | Purpose |
|---|---|---|
| `ResponseTextDeltaEvent` | `openai.types.responses` | One text token fragment |
| `ResponseFunctionCallArgumentsDeltaEvent` | `openai.types.responses` | One tool-argument JSON fragment |
| `ResponseOutputItemAddedEvent` | `openai.types.responses` | A new output item has started generating |
| `ResponseOutputItemDoneEvent` | `openai.types.responses` | An output item has finished generating |
| `ResponseReasoningSummaryTextDeltaEvent` | `openai.types.responses` | One reasoning **summary** token fragment (GPT-5 models only, and only when `summary=` is set) |
| `Reasoning` | `openai.types.shared` | Configures reasoning effort *and* summary generation on `ModelSettings` |
| `RawResponsesStreamEvent`, `RunItemStreamEvent` | `agents` | SDK-level wrappers used to tell raw events and run item events apart in the stream loop |
| `ItemHelpers` | `agents` | Convenience helpers for extracting text from completed items |

A note on reasoning specifically: OpenAI never exposes a reasoning model's actual internal chain-of-thought to the client, on any effort setting. What you can stream is an optional, separate **summary** of that thinking, and only if you explicitly request one with `summary=` on `Reasoning`. That's what `ResponseReasoningSummaryTextDeltaEvent` carries. You'll see this in action in Cell 10.

In [4]:
from openai.types.responses import (
    ResponseFunctionCallArgumentsDeltaEvent,
    ResponseOutputItemAddedEvent,
    ResponseOutputItemDoneEvent,
    ResponseReasoningSummaryTextDeltaEvent,
    ResponseTextDeltaEvent,
)
from openai.types.shared import Reasoning

from agents import (
    Agent,
    ItemHelpers,
    ModelSettings,
    RawResponsesStreamEvent,
    Runner,
    RunItemStreamEvent,
    function_tool,
)

## Cell 5 — Raw Events vs Run Item Events

Before writing any streaming code, it's worth understanding the two layers of events the SDK exposes through `result.stream_events()`.

| | Raw events (`RawResponsesStreamEvent`) | Run item events (`RunItemStreamEvent`) |
|---|---|---|
| Granularity | Token by token | One event per fully completed item |
| Timing | As they happen | Only once the item is fully done |
| Content | Responses API event objects | SDK `RunItem` wrappers |
| Best for | Streaming text to users in real time | Logging, UI state machines, tool tracking |
| Detection | `isinstance(event.data, SpecificType)` | `event.item.type` string check |

**Key raw event types you'll use in this notebook, all from `openai.types.responses`:**

- `ResponseTextDeltaEvent` — one text token
- `ResponseFunctionCallArgumentsDeltaEvent` — one tool-argument JSON fragment
- `ResponseOutputItemAddedEvent` — a new output item started
- `ResponseOutputItemDoneEvent` — an output item completed
- `ResponseReasoningSummaryTextDeltaEvent` — one reasoning **summary** token (GPT-5 models only, and only when a summary is requested)

Both raw events and run item events arrive on the *same* `stream_events()` loop. You don't choose one or the other for an entire run — you filter for the ones you care about, and it's common to use both together, which you'll see in Cell 11.

## Cell 6 — Observing the Full Raw Event Lifecycle

This cell runs a simple agent and prints the `type` string of every raw event that arrives, so you can see the actual Responses API protocol underneath `Runner.run_streamed()`.

A few things to note about the code:
- `Runner.run_streamed()` is **not awaited**. It returns a `RunResultStreaming` object immediately; the run happens as you iterate `result.stream_events()`.
- We filter with `event.type == "raw_response_event"` to isolate raw events from run item events for this first pass.
- `reasoning=Reasoning(effort="none")` keeps this run simple. GPT-5 models with reasoning `effort="none"` don't emit reasoning events, so you'll see a clean text-only lifecycle here.

For a short response, expect a sequence roughly like: `response.created`, `response.output_item.added`, `response.content_part.added`, several `response.output_text.delta` events (one per token), `response.content_part.done`, `response.output_item.done`, then `response.completed`.

Run this cell and count how many distinct event types you see for a five-word response.

In [5]:
agent = Agent(
    name="Lifecycle Agent",
    instructions="You are a helpful assistant. Be concise.",
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
)

result = Runner.run_streamed(agent, "Say: hello world")

print("All raw events:")
async for event in result.stream_events():
    if event.type == "raw_response_event":
      print(f" {event.type} {event.data.type}")

All raw events:
 raw_response_event response.created
 raw_response_event response.in_progress
 raw_response_event response.output_item.added
 raw_response_event response.content_part.added
 raw_response_event response.output_text.delta
 raw_response_event response.output_text.delta
 raw_response_event response.output_text.done
 raw_response_event response.content_part.done
 raw_response_event response.output_item.done
 raw_response_event response.completed


## Cell 7 — Streaming Text Deltas: the Readable Pattern

Now narrow the lifecycle down to the one event type most streaming UIs actually need: `ResponseTextDeltaEvent`.

- `isinstance(event.data, ResponseTextDeltaEvent)` is the correct detection pattern here. It gives you proper type narrowing, so your editor and type checker both know `event.data.delta` exists, instead of relying on a bare string comparison against `event.data.type`.
- `event.data.delta` is the token fragment string itself.
- `print(event.data.delta, end="", flush=True)` prints each fragment immediately next to the last one, rather than starting a new line per token.

This is the pattern you'll reach for whenever you want a chat UI to show the model "typing" its answer in real time.

In [6]:
result = Runner.run_streamed(
    agent,
    "Write a haiku about the ocean.",
)

print("Streaming token by token:")
async for event in result.stream_events():
    if (
        event.type == "raw_response_event"
        and isinstance(event.data, ResponseTextDeltaEvent)
    ):
        print(event.data.delta, end="", flush=True)
print()

Streaming token by token:
Salt breath on blue skin  
Waves fold the moon into glass  
Deep hush, endless tide


## Cell 8 — Run Item Events: the Semantic Milestone Pattern

Now switch to the other layer. Instead of watching individual tokens, you'll watch complete, meaningful milestones: a tool was called, a tool returned an output, a message was produced.

- `event.type == "run_item_stream_event"` filters for this layer.
- `event.name` is a semantic label such as `"tool_called"`, `"tool_output"`, or `"message_output_created"`. This makes routing logic readable without inspecting the underlying item shape.
- `event.item.type` tells you which kind of `RunItem` arrived: `tool_call_item`, `tool_call_output_item`, `message_output_item`, and so on.
- Items only appear here once they are **fully complete**. A `tool_call_item` fires the instant the model finishes emitting the call; a `tool_call_output_item` fires only after your tool function has actually executed and returned.
- `ItemHelpers.text_message_output(event.item)` is the shortcut for pulling the plain text out of a completed message item.

This is the layer you'd reach for to drive a UI state machine: show a spinner when `tool_called` fires, hide it when `tool_output` fires.

In [7]:
@function_tool
def analyse_sentiment(text: str) -> str:
    """Analyses the sentiment of the given text.

    Args:
        text: The text to analyse.
    """
    return "Sentiment: Positive (confidence: 0.92)"


sentiment_agent = Agent(
    name="Sentiment Agent",
    instructions=(
        "You are a sentiment analysis assistant. "
        "Use the analyse_sentiment tool."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[analyse_sentiment],
)

result = Runner.run_streamed(
    sentiment_agent,
    "Analyse: I absolutely love this product!",
)

async for event in result.stream_events():
    if event.type == "run_item_stream_event":
        name = event.name
        item_type = event.item.type
        print(f"[{name}] item type: {item_type}")
        if item_type == "tool_call_item":
            print(f"  Tool called: {event.item.tool_name}")
        elif item_type == "tool_call_output_item":
            print(f"  Tool output: {event.item.output}")
        elif item_type == "message_output_item":
            text = ItemHelpers.text_message_output(event.item)
            print(f"  Message: {text[:60]}")

[tool_called] item type: tool_call_item
  Tool called: analyse_sentiment
[tool_output] item type: tool_call_output_item
  Tool output: Sentiment: Positive (confidence: 0.92)
[message_output_created] item type: message_output_item
  Message: Positive (confidence: 0.92)


## Cell 9 — Streaming Function Call Argument Deltas in Real Time

This is one of the most practical raw-event patterns: watching the model build a tool call's JSON arguments token by token, before the call is even complete. The code below tracks a call through three phases: it starts, its arguments stream in, and it finishes. Let's walk through it phase by phase.

**The big picture.** The loop watches a tool call get built up in real time using three phases: started, arguments streaming in, and complete. It tracks the call using a dictionary, `function_calls`, keyed by `call_id`, so it would still work correctly even if the model made more than one tool call in the same turn.

**Filtering to raw events.** The loop sees both raw events and run item events on the same stream. Since this cell only cares about the raw layer, it opens with `if event.type != "raw_response_event": continue`, which skips anything that isn't a raw event.

**Phase 1 — a function call starts.** `response.output_item.added` fires whenever *any* new output item begins, and an output item could be a message, a function call, or a reasoning block. So the code checks `getattr(item, "type", None) == "function_call"` to filter for the case we care about. It uses `getattr` instead of `item.type` because `item` is typed as a broad union (`ResponseOutputItem`), and being defensive with `getattr` avoids an `AttributeError` if some variant doesn't carry that field. Once confirmed, it pulls the tool's `name` and its `call_id` (a unique ID for this specific call), opens a fresh entry in `function_calls` to accumulate the arguments string, and remembers `current_call_id` so the next phase knows which call the incoming deltas belong to.

**Phase 2 — arguments stream in.** As the model generates the tool call's JSON arguments, each fragment arrives as its own `ResponseFunctionCallArgumentsDeltaEvent`. `event.data.delta` is just a chunk of that JSON text, for example `{"tit`, then `le": "Q4`, and so on. The code appends each chunk onto the running string and prints it immediately, so you watch the JSON assemble live.

**Phase 3 — the call finishes.** `response.output_item.done` fires when any output item finishes. `hasattr(item, "call_id")` filters for function calls specifically, since a text message item wouldn't have a `call_id`. The code looks up the call in `function_calls`, prints a completion marker, and clears `current_call_id` if this was the one currently being tracked.

**Why the `call_id` bookkeeping matters.** If a model makes more than one tool call in a single turn, the delta events for each call could in principle interleave. Keying everything by `call_id`, rather than assuming "the current call," means each call's arguments accumulate in the right bucket even if that happens.

This pattern is genuinely useful in production: for a tool with a long argument list, showing the user a live preview of what the model is about to pass gives real, valuable feedback instead of a silent pause.

In [8]:
@function_tool
def create_report(
    title: str,
    summary: str,
    sections: list[str],
    priority: str,
) -> str:
    """Creates a structured report.

    Args:
        title: Report title.
        summary: Executive summary.
        sections: List of section names.
        priority: Priority level: low, medium, or high.
    """
    return f"Report '{title}' created with {len(sections)} sections."


report_agent = Agent(
    name="Report Agent",
    instructions=(
        "You are a report generation assistant. "
        "Use the create_report tool to generate reports."
    ),
    model=MODEL_NAME,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="none"),
        verbosity="low",
    ),
    tools=[create_report],
)

result = Runner.run_streamed(
    report_agent,
    "Create a high-priority report on Q4 sales with sections:"
    " Overview, Performance, Recommendations.",
)

function_calls: dict = {}
current_call_id = None

async for event in result.stream_events():
    if event.type != "raw_response_event":
        continue

    if event.data.type == "response.output_item.added":
        item = event.data.item
        if getattr(item, "type", None) == "function_call":
            name = getattr(item, "name", "unknown")
            call_id = getattr(item, "call_id", "unknown")
            function_calls[call_id] = {"name": name, "arguments": ""}
            current_call_id = call_id
            print(f"\n[CALL STARTED] {name}()")
            print("Arguments: ", end="", flush=True)

    elif isinstance(event.data, ResponseFunctionCallArgumentsDeltaEvent):
        if current_call_id and current_call_id in function_calls:
            function_calls[current_call_id]["arguments"] += event.data.delta
            print(event.data.delta, end="", flush=True)

    elif event.data.type == "response.output_item.done":
        item = event.data.item
        if hasattr(item, "call_id"):
            call_id = getattr(item, "call_id", "unknown")
            if call_id in function_calls:
                info = function_calls[call_id]
                print(f"\n[CALL COMPLETE] {info['name']}")
                if current_call_id == call_id:
                    current_call_id = None

print(f"\nFinal output: {result.final_output}")


[CALL STARTED] create_report()
Arguments: {"title":"Q4 Sales Report","summary":"High-priority report on Q4 sales.","sections":["Overview","Performance","Recommendations"],"priority":"high"}
[CALL COMPLETE] create_report

Final output: Report created: **Q4 Sales Report**.


## Cell 10 — Reasoning Summary Deltas: GPT-5 Models

GPT-5 reasoning models think internally before responding, but OpenAI never exposes that raw internal chain-of-thought to the client, on any effort setting. What you *can* stream is an optional, separate **summary** of that thinking, generated only when you explicitly ask for one. This cell streams that summary live, alongside the final answer. Let's walk through exactly how, since there are a few details that trip people up.

**Requesting a summary.** `Reasoning(effort="high", summary="detailed")` does two things at once. `effort` controls how much internal reasoning the model does. `summary` is the separate switch that tells OpenAI to generate a paraphrased summary of that reasoning *and* stream it back to you. Leaving out `summary=` means no summary is ever generated, no matter how high you set `effort`, and nothing will arrive on the event we're about to filter for.

**The correct event type.** The summary streams as `ResponseReasoningSummaryTextDeltaEvent`, with `event.data.type == "response.reasoning_summary_text.delta"`. This is a top-level import from `openai.types.responses`, no deep submodule path required. It's easy to confuse this with a similarly named event that only appears when routing through the SDK's Chat Completions compatibility layer, for example when talking to a third-party reasoning model. That one never fires for a direct OpenAI Responses API call like this, which is exactly what tripped this cell up the first time it was built.

**Switching models.** We use a dedicated `UPGRADED_MODEL` variable here rather than touching `MODEL_NAME`, since `gpt-5.4-mini` at low effort essentially never produces a meaningful reasoning summary. `MODEL_NAME` itself stays untouched for the rest of the notebook.

**Printing the two phases cleanly.** The loop uses two boolean flags, `thinking_started` and `answer_started`, so the `[THINKING]` and `[ANSWER]` labels print exactly once each, right before their first fragment, rather than being repeated on every delta. Watch the order the two phases print in: the summary streams first, and only once it's done does the final answer start appearing. That's the model thinking, then speaking, and now you can see the boundary happen live.

**Summary generation isn't guaranteed.** Even with `summary="detailed"` set, OpenAI decides per response whether there's a meaningful reasoning trace worth summarizing. For a short or simple question, you may occasionally see no summary at all, purely because the model didn't need much internal reasoning to answer. That's why the code checks `if not reasoning_text` at the end and prints an explicit note instead of silently showing nothing. This is a genuine characteristic of the feature, not a bug in your code, so a harder, multi-step question (like the one used here) makes a visible summary far more likely, but never fully guaranteed.

In [9]:
UPGRADED_MODEL = "gpt-5.5"

reasoning_agent = Agent(
    name="Reasoning Agent",
    instructions="You are a helpful assistant that thinks carefully and explains your reasoning.",
    model=UPGRADED_MODEL,
    model_settings=ModelSettings(
        reasoning=Reasoning(effort="high", summary="detailed"),
        verbosity="low",
    ),
)

result = Runner.run_streamed(
    reasoning_agent,
    "A train leaves at 2pm going 60mph. A second train leaves the same "
    "station 30 minutes later going 75mph on the same track. "
    "At what time does the second train catch up? Explain your reasoning.",
)

reasoning_text = ""
output_text = ""
thinking_started = False
answer_started = False

async for event in result.stream_events():
    if event.type == "raw_response_event":
        if isinstance(event.data, ResponseReasoningSummaryTextDeltaEvent):
            if not thinking_started:
                print("[THINKING] ", end="", flush=True)
                thinking_started = True
            reasoning_text += event.data.delta
            print(event.data.delta, end="", flush=True)
        elif isinstance(event.data, ResponseTextDeltaEvent):
            if not answer_started:
                print("\n\n[ANSWER] ", end="", flush=True)
                answer_started = True
            output_text += event.data.delta
            print(event.data.delta, end="", flush=True)

if not reasoning_text:
    print(
        "(No reasoning summary was returned for this run. Summary generation "
        "isn't guaranteed on every call, even with summary='detailed' set.)"
    )

print(f"\n\nReasoning chars collected: {len(reasoning_text)}")
print(f"Final output: {result.final_output}")

[THINKING] **Calculating train speeds**

I need to answer with some reasoning about relative speeds. So, Train 1 leaves at 2 PM, traveling at 60 mph, and by 2:30 PM, it has a 30-mile lead. Train 2 travels at 75 mph, giving a relative speed of 15 mph. To catch up the 30 miles, it will take 2 hours after 2:30 PM, which means it catches up at 4:30 PM. It could be a weird scenario, especially if they’re on the same track!

[ANSWER] By 2:30pm, the first train has been traveling for 30 minutes at 60 mph:

- \(60 \times 0.5 = 30\) miles ahead

The second train travels 75 mph, so it gains on the first train at:

- \(75 - 60 = 15\) mph

Time to close the 30-mile gap:

- \(30 \div 15 = 2\) hours

So the second train catches up 2 hours after 2:30pm:

**4:30pm**.

Reasoning chars collected: 410
Final output: By 2:30pm, the first train has been traveling for 30 minutes at 60 mph:

- \(60 \times 0.5 = 30\) miles ahead

The second train travels 75 mph, so it gains on the first train at:

- \(75 - 60 

## Cell 11 — The Combined Production Pattern

This is the pattern most production apps actually ship: raw events for streaming text to the user in real time, and run item events for semantic milestones, both handled in the same `stream_events()` loop.

Notice the structure: the `if`/`elif` at the top level branches on `event.type` first (`"raw_response_event"` vs `"run_item_stream_event"`), then narrows further inside each branch. You only handle the events you actually care about and skip everything else, which is exactly what you'd do in a real chat application.

In [10]:
result = Runner.run_streamed(
    sentiment_agent,
    "Analyse: This is exactly what I needed!",
)

async for event in result.stream_events():
    if event.type == "raw_response_event":
        if isinstance(event.data, ResponseTextDeltaEvent):
            print(event.data.delta, end="", flush=True)
    elif event.type == "run_item_stream_event":
        if event.item.type == "tool_call_item":
            print(f"\n[TOOL: {event.item.tool_name}]", flush=True)
        elif event.item.type == "tool_call_output_item":
            print(f"[RESULT: {event.item.output}]", flush=True)

print()
print("Final:", result.final_output)


[TOOL: analyse_sentiment]
[RESULT: Sentiment: Positive (confidence: 0.92)]
Positive (0.92)
Final: Positive (0.92)


## Cell 12 — Raw Event Decision Guide

Four patterns cover almost every streaming use case you'll build with this SDK:

**Pattern A: Stream text to the user in real time**
Use: `isinstance(event.data, ResponseTextDeltaEvent)`
When: chat UI, token-by-token display

**Pattern B: Track semantic milestones**
Use: `RunItemStreamEvent` with `event.name` filtering
When: logging, UI state machines, progress indicators

**Pattern C: Track tool argument generation**
Use: `isinstance(event.data, ResponseFunctionCallArgumentsDeltaEvent)`
When: real-time tool parameter preview

**Pattern D: Track reasoning summaries (GPT-5 only, requires `summary=` set)**
Use: `isinstance(event.data, ResponseReasoningSummaryTextDeltaEvent)`
When: "thinking..." UI indicators

**Rule of thumb:** if a human is watching, use raw events. They give you the fastest feedback. If your code is reacting, use run item events. They give you complete, type-safe objects. Most production apps need both, exactly as you saw in Cell 11.

## Cell 13 — Raw Event Type Reference

A single table to bookmark for the raw event types covered in this notebook, all importable from the top-level `openai.types.responses` package:

| Class | `event.data.type` string | Key field | When it arrives |
|---|---|---|---|
| `ResponseTextDeltaEvent` | `"response.output_text.delta"` | `.delta: str` | Each text token |
| `ResponseFunctionCallArgumentsDeltaEvent` | `"response.function_call_arguments.delta"` | `.delta: str` | Each argument JSON fragment |
| `ResponseReasoningSummaryTextDeltaEvent`* | `"response.reasoning_summary_text.delta"` | `.delta: str` | Each reasoning summary token (GPT-5, requires `summary=` set on `Reasoning`) |
| `ResponseOutputItemAddedEvent` | `"response.output_item.added"` | `.item` | New output item started |
| `ResponseOutputItemDoneEvent` | `"response.output_item.done"` | `.item` | Output item completed |

\* This carries a paraphrased *summary* of the model's reasoning, never its raw internal chain-of-thought, and only streams when `summary=` is explicitly set on `Reasoning`. Generation isn't guaranteed even then.

That's the full raw event toolkit for this lecture. Lecture 4.4 picks up with `RunConfig`, where you'll control run-wide settings like tracing and turn limits.